# Blood Cell Detection with YOLOv8 (BCCD Dataset)
**Applied Machine Learning — Swinburne University**

Fine-tuned YOLOv8n on the BCCD (Blood Cell Count and Detection) dataset to detect **RBC**, **WBC**, and **Platelets** in microscope images, improving mAP50 from ~0.00 (COCO-pretrained baseline) to **0.87** on the held-out test set.

**Dataset used:** [BCCD (Blood Cell Count and Detection) Dataset](https://public.roboflow.com/object-detection/bccd) — Roboflow Public Datasets
- 874 images total (765 train / 73 valid / 36 test), 3 classes: `RBC` (Red Blood Cell), `WBC` (White Blood Cell), `Platelets`
- License: MIT
- Pre-split into train / valid / test by Roboflow
- Download (YOLOv8 format): https://public.roboflow.com/object-detection/bccd/3/download/yolov8

**Workflow:**
1. Upload & extract the dataset zip
2. Inspect folder structure and `data.yaml`
3. Evaluate a **pretrained** model on the test set (baseline)
4. **Fine-tune** the model on the dataset
5. Evaluate the **fine-tuned** model on the same test set
6. Compare pretrained vs fine-tuned results
7. Visualize predictions on sample test images


## Step 1 — Upload the dataset
Go to the dataset page, choose **YOLOv8** format and download the zip:
https://public.roboflow.com/object-detection/bccd/3/download/yolov8

Then upload that zip file below (no Roboflow account/API key needed for this public dataset).

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile
import os

# Grab whatever filename was uploaded (avoids hardcoding a name that may
# differ if you re-upload, e.g. Colab appending "(1)" to duplicate names).
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall("dataset")

print("Extracted:", zip_name)

## Step 2 — Inspect the dataset structure and annotation format

In [ ]:
import os

for root, dirs, files_ in os.walk("dataset"):
    level = root.replace("dataset", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level >= 1:  # don't list every single file, just peek
        for f in files_[:3]:
            print(f"{indent}  {f}")
        if len(files_) > 3:
            print(f"{indent}  ... ({len(files_)} files total)")

In [ ]:
# Look at the data.yaml (class names, train/valid/test paths)
with open("dataset/data.yaml", "r") as f:
    print(f.read())

In [ ]:
# Peek at one YOLO-format label file (class_id x_center y_center width height, all normalised 0-1)
labels_dir = "dataset/train/labels"
sample_label = os.listdir(labels_dir)[0]
with open(os.path.join(labels_dir, sample_label)) as f:
    print(f"Sample label file: {sample_label}\n")
    print(f.read())

## Step 3 — Install Ultralytics

In [ ]:
!pip install ultralytics -q

## Step 4 — Evaluate the PRETRAINED model (baseline)
Loading a COCO-pretrained model and evaluating it directly on our test set, *before* any fine-tuning.
Note: BCCD's classes (RBC, WBC, Platelets) aren't COCO classes, so we expect this baseline to perform poorly —
that's expected and is exactly what we're measuring.

In [ ]:
from ultralytics import YOLO

MODEL_NAME = "yolov8n.pt"  # swap for "yolo26n.pt" if you want to try the newer YOLO26 nano model

pretrained_model = YOLO(MODEL_NAME)

In [ ]:
baseline_metrics = pretrained_model.val(data="dataset/data.yaml", split="test")

print("=== PRETRAINED MODEL (baseline) — TEST SET ===")
print(f"mAP50:    {baseline_metrics.box.map50:.4f}")
print(f"mAP50-95: {baseline_metrics.box.map:.4f}")
print(f"Precision:{baseline_metrics.box.mp:.4f}")
print(f"Recall:   {baseline_metrics.box.mr:.4f}")

## Step 5 — Fine-tune the model on BCCD
`freeze=10` freezes the first 10 layers (the backbone), so only the head is retrained —
this is faster and works well on small datasets like this one.

In [ ]:
model = YOLO(MODEL_NAME)  # fresh copy so the baseline model above stays untouched

results = model.train(
    data="dataset/data.yaml",
    epochs=30,
    imgsz=640,
    freeze=10
)

## Step 6 — Evaluate the FINE-TUNED model on the same test set

In [ ]:
finetuned_metrics = model.val(data="dataset/data.yaml", split="test")

print("=== FINE-TUNED MODEL — TEST SET ===")
print(f"mAP50:    {finetuned_metrics.box.map50:.4f}")
print(f"mAP50-95: {finetuned_metrics.box.map:.4f}")
print(f"Precision:{finetuned_metrics.box.mp:.4f}")
print(f"Recall:   {finetuned_metrics.box.mr:.4f}")

## Step 7 — Compare pretrained vs fine-tuned

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Metric": ["mAP50", "mAP50-95", "Precision", "Recall"],
    "Pretrained": [
        baseline_metrics.box.map50,
        baseline_metrics.box.map,
        baseline_metrics.box.mp,
        baseline_metrics.box.mr,
    ],
    "Fine-tuned": [
        finetuned_metrics.box.map50,
        finetuned_metrics.box.map,
        finetuned_metrics.box.mp,
        finetuned_metrics.box.mr,
    ],
})
comparison["Improvement"] = comparison["Fine-tuned"] - comparison["Pretrained"]
comparison

## Step 8 — Visualize predictions on sample test images
Quick sanity check: run the fine-tuned model on a few test images and plot the predicted boxes.

In [ ]:
import matplotlib.pyplot as plt
import glob

test_images = glob.glob("dataset/test/images/*.jpg")[:6]
results_vis = model.predict(test_images, conf=0.25)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, r in zip(axes.flatten(), results_vis):
    ax.imshow(r.plot()[:, :, ::-1])  # BGR -> RGB
    ax.axis("off")
plt.tight_layout()
plt.savefig("sample_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

## Results Summary

| Metric | Pretrained (COCO, no fine-tuning) | Fine-tuned (30 epochs, frozen backbone) |
|---|---|---|
| mAP50 | ~0.00 | **0.87** |
| mAP50-95 | ~0.00 | **0.60** |
| Precision | ~0.00 | **0.79** |
| Recall | ~0.01 | **0.88** |

**Per-class mAP50 (fine-tuned, test set):** WBC 0.97 · RBC 0.87 · Platelets 0.78

The pretrained COCO model scores near-zero because RBC/WBC/Platelets aren't COCO classes — this confirms fine-tuning, not the base model, is doing the work. WBC is easiest to detect (large, distinct nucleus); Platelets are hardest (small, low-contrast, easy to miss or double-count).

In [ ]:
# Best fine-tuned weights, saved automatically by Ultralytics
best_weights = model.trainer.best
print("Best fine-tuned weights saved at:", best_weights)